## Homework 3: Pranav Prakash
2026 lead-scoring dataset

In [66]:
%pip install -q kagglehub pandas
import pandas as pd
from pathlib import Path
from urllib.request import urlretrieve

url = (
    "https://raw.githubusercontent.com/DataTalksClub/"
    "machine-learning-zoomcamp/main/cohorts/2026/data/"
    "course_lead_scoring_2026.csv"
)

file_path = Path("course_lead_scoring_2026.csv")

if not file_path.exists():
    urlretrieve(url, file_path)

df = pd.read_csv(file_path)

df.head()

Note: you may need to restart the kernel to use updated packages.


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [67]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [68]:
df.columns
df.isnull().sum()
df.dtypes
# look at the distince values in each column
df.nunique()
# look at the count of distinct values in each column including the values 
df.value_counts()

df.iloc[:100]

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
...,...,...,...,...,...,...,...,...,...
95,paid_ads,retail,unemployed,asia,21084.0,4,9,0.68,1
96,referral,finance,employed,asia,75202.0,2,5,0.59,1
97,events,technology,unemployed,europe,50096.0,1,0,0.18,0
98,organic_search,technology,employed,europe,95649.0,1,1,0.37,1


In [69]:
## Define the column names into variable types
categorical = list(df.columns[df.dtypes == 'str'])
numerical = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

df[categorical] = df[categorical].fillna('NA')
df[categorical].iloc[:100]
df[numerical] = df[numerical].fillna(0)
df[numerical].iloc[:100]

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,88160.0,3,4,0.64,1
1,72688.0,3,7,0.72,1
2,44697.0,3,4,0.58,1
3,0.0,3,6,0.57,0
4,24062.0,4,5,0.62,1
...,...,...,...,...,...
95,21084.0,4,9,0.68,1
96,75202.0,2,5,0.59,1
97,50096.0,1,0,0.18,0
98,95649.0,1,1,0.37,1


In [70]:
## Question 1
df.industry.mode()

0    technology
Name: industry, dtype: str

In [71]:
## Correlation Matrix
# feature-to-feature correlation, to check redundancy
df[numerical].corr()

#interaction_count and lead_score 0.915746
#number_of_courses_viewed and lead_score 0.757204
#number_of_courses_viewed and interaction_count 0.721609
#annual_income and interaction_count 0.122842

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [72]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

len(df), len(df_train), len(df_val), len(df_test)

(5000, 3000, 1000, 1000)

In [73]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [74]:
# setting up the target variable for the exercise
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

df_train = df_train.drop(columns = ['converted'])
df_val = df_val.drop(columns = ['converted'])
df_test = df_test.drop(columns = ['converted'])

In [75]:
## Question 3
from sklearn.metrics import mutual_info_score

for c in categorical: 
    print (c)
    score = (mutual_info_score(df_full_train[c],df_full_train['converted']))
    print(round(score, 2))

lead_source
0.03
industry
0.0
employment_status
0.02
location
0.0


In [76]:
from IPython import display
for c in categorical:     
    print(c)
    global_conversion_rate = df_full_train.converted.mean()
    df_group = df_full_train.groupby(c)['converted'].agg(['mean','count'])
    df_group['diff'] = df_group['mean'] - global_conversion_rate
    df_group['risk'] = df_group['mean'] / global_conversion_rate
    display.display(df_group)
    print()
    print()

lead_source


,mean,count,diff,risk
lead_source,,,,
NA,0.610169,118,0.039419,1.069066
events,0.411290,496,-0.159460,0.720614
organic_search,0.677579,1008,0.106829,1.187174
paid_ads,0.427957,930,-0.142793,0.749815
referral,0.763844,614,0.193094,1.338316
social_media,0.547962,834,-0.022788,0.960073




industry


,mean,count,diff,risk
industry,,,,
NA,0.587629,194,0.016879,1.029573
education,0.500000,518,-0.070750,0.876040
finance,0.613402,582,0.042652,1.074730
healthcare,0.559398,665,-0.011352,0.980111
manufacturing,0.548969,388,-0.021781,0.961838
retail,0.553741,735,-0.017009,0.970200
technology,0.611111,918,0.040361,1.070716




employment_status


,mean,count,diff,risk
employment_status,,,,
NA,0.580247,162,0.009497,1.016639
employed,0.656459,2090,0.085709,1.150170
self_employed,0.548276,580,-0.022474,0.960623
student,0.419773,617,-0.150977,0.735476
unemployed,0.435572,551,-0.135178,0.763157




location


,mean,count,diff,risk
location,,,,
NA,0.603550,169,0.032800,1.057469
africa,0.546154,390,-0.024596,0.956906
asia,0.585308,844,0.014558,1.025507
europe,0.584821,896,0.014071,1.024654
north_america,0.576923,1092,0.006173,1.010816
south_america,0.525452,609,-0.045298,0.920633


## One-Hot Encoding

In [77]:
from sklearn.feature_extraction import DictVectorizer

In [78]:
categorical = df_train.select_dtypes(include=['object']).columns.tolist()
numerical = df_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

##This step converts the training data from a pandas DataFrame into a list of dictionaries, 
# which is the input format DictVectorizer expects.
train_dicts = df_train[categorical + numerical].to_dict(orient = 'records')


/tmp/ipykernel_22762/2887871356.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = df_train.select_dtypes(include=['object']).columns.tolist()


In [79]:
dv = DictVectorizer(sparse=False) # sparse=False means we want a dense array as output
dv.fit(train_dicts) # fit the training data for the purposes of one-hot-encoding
#print(dv.get_feature_names_out()) # get the new feature names for variables
# transform the training data using the fitted DictVectorizer, converting the dictionaries into a numerical array.
dv.transform(train_dicts) 
# Get feature matrix
X_train = dv.fit_transform(train_dicts)





## Key point to remember: 
Dictvectorizer is only trained on the training df, not the validation df. 
We will only transform the val_dicts to get the feature matrix. 

In [80]:
# Treating the validation dataset
val_dicts = df_val[categorical + numerical].to_dict(orient = 'records')
X_val = dv.transform(val_dicts)
X_val

array([[8.0457e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        3.0000e+00],
       [7.4480e+04, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00],
       [4.3944e+04, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        2.0000e+00],
       ...,
       [6.2439e+04, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00],
       [4.0878e+04, 0.0000e+00, 0.0000e+00, ..., 1.0000e+00, 0.0000e+00,
        1.0000e+00],
       [4.8769e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        3.0000e+00]], shape=(1000, 28))

## What do we have so far? 
1. Loaded the dataset, prelimiary EDA and data cleaning
2. Split the data into train, val, test using scikit-learn (train_test_split)
3. Identified the target variable 'converted' and isolated that into a 'y' variable for train, val and test
4. Dropped target variable from the main df
5. One-hot encoding using Dictvectorizer, for all the categorical and numerical variables
6. Dictvectorizer is trained using df_train, but transforms both train and val, to get feature matrix for both
7. Next step is to train the logistic regression

In [81]:
from sklearn.linear_model import LogisticRegression

## UNDER THE HOOD this is how it works
def sigmoid (z): 
    return 1 / (1 + np.exp(-z))

def logistic_regression(xi): 
    score = w0

    for j in range(len(xi)):
        score += w[j] * xi[j]
    result = sigmoid(score)
    return result

In [82]:
#Fit the logistic regression model using the training dataset. 
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train,y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [83]:
## Once the model is fit, find the intercept and weights

model.intercept_[0] #bias term / intercept 
model.coef_[0].round(3) #weights

array([-0.   , -0.001,  0.007, -0.003, -0.008, -0.006, -0.   , -0.003,
       -0.001, -0.002, -0.002, -0.002, -0.001,  0.137,  0.011,  0.   ,
       -0.009,  0.007, -0.014,  0.009, -0.004,  0.   , -0.002, -0.002,
       -0.001, -0.003, -0.004,  0.054])

# Model Interpretation

In [84]:
## creating a combined view of the feature names & respective weights
list(zip(dv.get_feature_names_out(),model.coef_[0].round(3)))

[('annual_income', np.float64(-0.0)),
 ('employment_status=NA', np.float64(-0.001)),
 ('employment_status=employed', np.float64(0.007)),
 ('employment_status=self_employed', np.float64(-0.003)),
 ('employment_status=student', np.float64(-0.008)),
 ('employment_status=unemployed', np.float64(-0.006)),
 ('industry=NA', np.float64(-0.0)),
 ('industry=education', np.float64(-0.003)),
 ('industry=finance', np.float64(-0.001)),
 ('industry=healthcare', np.float64(-0.002)),
 ('industry=manufacturing', np.float64(-0.002)),
 ('industry=retail', np.float64(-0.002)),
 ('industry=technology', np.float64(-0.001)),
 ('interaction_count', np.float64(0.137)),
 ('lead_score', np.float64(0.011)),
 ('lead_source=NA', np.float64(0.0)),
 ('lead_source=events', np.float64(-0.009)),
 ('lead_source=organic_search', np.float64(0.007)),
 ('lead_source=paid_ads', np.float64(-0.014)),
 ('lead_source=referral', np.float64(0.009)),
 ('lead_source=social_media', np.float64(-0.004)),
 ('location=NA', np.float64(0.0))

## Question 4

In [85]:
## Using the model, make a prediction on the validation dataset. call it y_pred
y_pred = model.predict_proba(X_val)[:,1]
y_pred
conversion_decision = (y_pred >= 0.5)

##compare this to the y target variable from the validation dataset 
## mean works because of the binary variables
accuracy = round((y_val == conversion_decision).mean(),2)
accuracy


np.float64(0.64)

## Question 5
removing interaction_count has the biggest drop in accuracy

In [86]:
numerical = ['annual_income','number_of_courses_viewed','lead_score']

categorical = ['lead_source','industry', 'employment_status', 'location']

train_dicts = df_train[categorical + numerical].to_dict(orient = 'records')
dv = DictVectorizer(sparse=False) # sparse=False means we want a dense array as output
dv.fit(train_dicts) # fit the training data for the purposes of one-hot-encoding
#print(dv.get_feature_names_out()) # get the new feature names for variables
# transform the training data using the fitted DictVectorizer, converting the dictionaries into a numerical array.
dv.transform(train_dicts) 
# Get feature matrix
X_train = dv.fit_transform(train_dicts)

# Treating the validation dataset
val_dicts = df_val[categorical + numerical].to_dict(orient = 'records')
X_val = dv.transform(val_dicts)

#Fit the logistic regression model using the training dataset. 
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train,y_train)
## Using the model, make a prediction on the validation dataset. call it y_pred
y_pred = model.predict_proba(X_val)[:,1]
conversion_decision = (y_pred >= 0.5)

##compare this to the y target variable from the validation dataset 
## mean works because of the binary variables
accuracy = (y_val == conversion_decision).mean()
accuracy


np.float64(0.601)

# Question 6

In [87]:
categorical = df_train.select_dtypes(include=['object']).columns.tolist()
numerical = df_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
train_dicts = df_train[categorical + numerical].to_dict(orient = 'records')
val_dicts = df_val[categorical + numerical].to_dict(orient = 'records')
C = [0.000001, 0.00001, 0.0001, 0.001]
dv = DictVectorizer(sparse=False) 
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

for c in C: 
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train,y_train)
    y_pred = model.predict_proba(X_val)[:,1]
    conversion_decision = (y_pred >= 0.5)
    accuracy = (y_val == conversion_decision).mean()
    print(f"C = {c:.6f} | Validation accuracy = {accuracy:.2%}")


C = 0.000001 | Validation accuracy = 59.80%
C = 0.000010 | Validation accuracy = 59.80%
C = 0.000100 | Validation accuracy = 61.30%
C = 0.001000 | Validation accuracy = 64.50%


/tmp/ipykernel_22762/707810663.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = df_train.select_dtypes(include=['object']).columns.tolist()


## Model Interpretation, Training a smaller model

In [99]:
numerical = ['number_of_courses_viewed','lead_score']

categorical = ['lead_source', 'employment_status']

train_dicts = df_train[categorical + numerical].to_dict(orient = 'records')
dv = DictVectorizer(sparse=False) 
dv.fit(train_dicts) # fit the training data for the purposes of one-hot-encoding


dv.transform(train_dicts) 
# Get feature matrix
X_train = dv.fit_transform(train_dicts)

# Treating the validation dataset
val_dicts = df_val[categorical + numerical].to_dict(orient = 'records')
X_val = dv.transform(val_dicts)

#Fit the logistic regression model using the training dataset. 
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train,y_train)
## Using the model, make a prediction on the validation dataset. call it y_pred
y_pred = model.predict_proba(X_val)[:,1]
conversion_decision = (y_pred >= 0.5)

##compare this to the y target variable from the validation dataset 
## mean works because of the binary variables
accuracy = (y_val == conversion_decision).mean()
accuracy


np.float64(0.727)

In [97]:
list(zip(dv.get_feature_names_out(),model.coef_[0].round(3)))
w0 = model.intercept_[0]
w = model.coef_[0].round(3) 

w0,w



(np.float64(-1.6980605535428),
 array([-0.237, -0.069, -0.373, -0.73 , -0.289,  4.398, -0.191, -0.495,
        -0.1  , -0.519, -0.012, -0.381,  0.216]))

In [98]:
list(zip(dv.get_feature_names_out(),model.coef_[0].round(3)))

[('employment_status=NA', np.float64(-0.237)),
 ('employment_status=employed', np.float64(-0.069)),
 ('employment_status=self_employed', np.float64(-0.373)),
 ('employment_status=student', np.float64(-0.73)),
 ('employment_status=unemployed', np.float64(-0.289)),
 ('lead_score', np.float64(4.398)),
 ('lead_source=NA', np.float64(-0.191)),
 ('lead_source=events', np.float64(-0.495)),
 ('lead_source=organic_search', np.float64(-0.1)),
 ('lead_source=paid_ads', np.float64(-0.519)),
 ('lead_source=referral', np.float64(-0.012)),
 ('lead_source=social_media', np.float64(-0.381)),
 ('number_of_courses_viewed', np.float64(0.216))]

In [100]:
y_full_train = df_full_train.converted.values

## Using the model: train on the full training set, evaluate on test

### Where this step fits
Earlier, we split the data three ways: **train / validation / test**. We used train + validation to make every modelling decision: which features to use, which value of `C`, which threshold. Those decisions are now **frozen**. This step answers one question:

> How well will the final model perform on data it has never seen?

### Why retrain on `df_full_train`?
`df_full_train` = train + validation. Once validation has done its job (helping us choose the model), holding it back only wastes data. More data usually means slightly better coefficients, so we refit the **same pipeline with the same hyperparameters** on the combined set.

The test set has not been used for any decision so far. That is what makes its score an honest estimate of real-world performance.

### The systematic approach: two phases
Think of this as a pipeline. Whatever the training data goes through in Phase 1 is **replayed, not relearned**, on the test data in Phase 2.

**Phase 1: Fit (full train)**
1. **Select features.** Take `categorical + numerical` only. The target (`churn`) must not be included.
2. **Convert to dictionaries.** `to_dict(orient='records')` gives one dict per customer, which is the format `DictVectorizer` expects.
3. **Fit + transform the vectorizer.**
   - *fit*: learns the vocabulary. Every category value becomes its own one-hot column.
   - *transform*: builds the feature matrix `X_full_train`.
4. **Fit the model.** Train `LogisticRegression` on `X_full_train` and `y_full_train`, using the hyperparameters chosen during validation.

**Phase 2: Apply (test)**

5. **Select the same features** from `df_test` and convert them to dictionaries.
6. **`transform` only, never `fit`.** Use the `dv` that was already fitted. Refitting on test would:
   - leak test information into the pipeline, and
   - possibly produce a different set or order of columns, so the model's coefficients would line up with the wrong features.

   A category that appears only in test just gets all zeros. The model treats it as "none of the categories I know."
7. **Predict probabilities.** `predict_proba(X_test)[:, 1]` gives P(churn = 1). Column 0 is P(churn = 0).
8. **Apply the threshold.** `y_pred >= 0.5` turns probabilities into churn / no-churn decisions.
9. **Score.** `(churn_decision == y_test).mean()` is **accuracy**: the fraction of customers classified correctly.

### Rule of thumb
> Anything that is *fit* learns from training data only. This includes the vectorizer, the model, scalers, and imputation values. Test data only ever goes through `.transform()` / `.predict()`.

Quick sanity check: `X_test.shape[1] == X_full_train.shape[1]`. The columns must match.

### Interpreting the result
Compare test accuracy with the validation accuracy from earlier.
- **Close (within a percent or two):** the model generalizes, and we didn't overfit to the validation set while tuning.
- **Noticeably lower:** possible causes are overfitting to validation, data leakage, or the splits differing in distribution. Investigate before trusting the model.

In [110]:
## Using the model

full_train_dicts = df_full_train[categorical + numerical].to_dict(orient = 'records')
dv = DictVectorizer(sparse=False) 
dv.fit(full_train_dicts) # fit the training data for the purposes of one-hot-encoding

# Get feature matrix
X_full_train = dv.fit_transform(full_train_dicts)

#Fit the logistic regression model using the training dataset. 
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_full_train,y_full_train)

## Repeat step for test
test_dicts = df_test[categorical + numerical].to_dict(orient = 'records')
X_test = dv.transform(test_dicts) #no need to fit the model on the test dataset

y_pred = model.predict_proba(X_test)[:,1]
churn_decision = (y_pred >= 0.5)

(churn_decision == y_test).mean()

np.float64(0.728)

In [112]:
## using with a customer example

customer = test_dicts[15]
customer

{'lead_source': 'paid_ads',
 'employment_status': 'unemployed',
 'number_of_courses_viewed': 1,
 'lead_score': 0.4}

In [118]:
X_small = dv.transform([customer])
X_small
model.predict_proba(X_small)[0,1]

np.float64(0.36876553726425976)

In [ ]:
y_test[15] #check in the converted column of the test dataset to confirm. Looks like the model was wrong. 

np.int64(1)